# Predikcija popularnosti pesama na Spotify-u

**Autori:** Isidora Pavlović E2 131/2025, Nikola Spasojević E2 82/2025

## 1. Pregled i eksplorativna analiza podataka

Dataset sadrži 114,000 pesama iz 125 žanrova sa sledećim obeležjima:

**Audio karakteristike (13):**
- `danceability` - pogodnost za ples
- `energy` - energičnost
- `loudness` - glasnoća
- `speechiness` - prisustvo govora
- `acousticness` - akustičnost
- `instrumentalness` - instrumentalnost
- `liveness` - prisustvo publike
- `valence` - valentnost (pozitivnost)
- `tempo` - tempo
- `duration_ms` - trajanje u milisekundama
- `key` - tonalitet
- `mode` - modalitet (dur/mol)
- `time_signature` - takt

**Metapodaci:**
- `track_name`, `artists`, `album_name`
- `track_genre` - muzički žanr
- `explicit` - da li pesma ima eksplicitan sadržaj

**Ciljno obeležje:**
- `popularity` - popularnost pesme (0-100), izračunata Spotify algoritmom na osnovu broja reprodukcija i njihove skorašnjosti

In [ ]:
# Učitavanje podataka
df = pd.read_csv('dataset.csv')
df = df.dropna(subset=['artists', 'album_name', 'track_name'])

print(f"Ukupno pesama: {len(df):,}")
print(f"Broj žanrova: {df['track_genre'].nunique()}")

## 2. Ključna otkrića iz analize

### 2.1 Distribucija popularnosti

Analiza je pokazala da:
- **Većina pesama ima nisku popularnost** - distribucija je nagnuta levo
- Srednja vrednost: ~33, medijana: ~28
- Veliki deo pesama (>40%) ima popularnost ispod 25

### 2.2 Korelacija sa audio karakteristikama

Najjače korelacije sa popularnošću:
- **Pozitivne:** `loudness` (+0.27), audio karakteristike visokog energetskog nivoa
- **Negativne:** `acousticness` (-0.21), `instrumentalness` (-0.18)

**Zakljucak:** Sve korelacije su slabe (|r| < 0.3), što ukazuje da popularnost zavisi od mnogo drugih faktora

### 2.3 Uticaj žanra

ANOVA test: **Žanr značajno utiče na popularnost** (p < 0.001)
- Različiti žanrovi imaju različite prosečne popularnosti
- Jazz, klasična muzika - niža popularnost
- Pop, dance - viša popularnost

In [ ]:
![Distribucija popularnosti](viz_1_distribucija_popularnosti.png)

**Napomena:** Sve vizualizacije u ovom notebook-u su generisane pomoću `generate_visualizations.py` skripte

### 2.4 Analiza šablona i klastera

Pored analize pojedinačnih atributa, važno je identifikovati **dublje šablone** i **klastere** u podacima. Primenjujemo K-means klasterizaciju da otkrijemo grupe pesama sa sličnim karakteristikama i analiziramo kako se klasteri razlikuju po popularnosti.

**Metodologija:**

1. **K-means algoritam** - deli podatke u grupe na osnovu sličnosti audio karakteristika
2. **StandardScaler normalizacija** - sve karakteristike dobijaju jednaku težinu (mean=0, std=1)
3. **PCA (Principal Component Analysis)** - smanjuje dimenzionalnost sa 7D na 2D za vizualizaciju
4. **Analiza profila** - svaki klaster ima specifičan "profil" audio karakteristika
5. **Analiza interakcija** - ispitivanje kako features zajedno utiču na popularnost

Umesto fokusa na pojedinačne korelacije, otkrivamo **kombinacije karakteristika** koje definišu uspešne šablone.

In [ ]:
# K-means klasterizacija
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Normalizacija audio features
audio_cols = ['danceability', 'energy', 'loudness', 'acousticness', 
              'instrumentalness', 'valence', 'tempo']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[audio_cols])

# K-means sa 4 klastera
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_scaled)

# Analiza klastera
cluster_stats = df.groupby('cluster').agg({
    'popularity': ['mean', 'median', 'count'],
    'danceability': 'mean',
    'energy': 'mean',
    'loudness': 'mean',
    'acousticness': 'mean'
}).round(2)

print("Karakteristike klastera:")
print(cluster_stats)
print(f"\nPronadjena su {df['cluster'].nunique()} klastera pesama")

In [ ]:
![Vizualizacija klastera](viz_2_klasteri.png)

**Napomena o PCA:**
- Smanjuje dimenzionalnost sa 7 audio features na 2 dimenzije za vizualizaciju
- Prve 2 glavne komponente obuhvataju najveći deo varijabilnosti podataka
- Omogućava 2D prikaz višedimenzionalnih klastera

**Otkriveni šabloni:**

Klaster analiza identifikovala je **4 distinktne grupe pesama**:

- Klaster 0 - **Akustične:** visoka acousticness, niska energy/loudness (mirne kompozicije)
- Klaster 1 - **Energične:** visoka energy + loudness + danceability (dinamične pesme, **najveća popularnost**)
- Klaster 2 - **Instrumentalne:** visoka instrumentalness (bez vokala, **najniža popularnost**)
- Klaster 3 - **Balansirana:** umerene vrednosti svih atributa (mainstream muzika)

**Zašto je ovo važno?**

Umesto: *"Loudness korelira +0.27 sa popularnošću"*, otkrivamo: *"Pesme koje **kombinuju** visoku energy + loudness + danceability formiraju klaster sa **značajno većom** popularnošću"*

**Zaključak:** Popularnost zavisi od **šablona** (kombinacija karakteristika), ne pojedinačnih atributa. Različiti šabloni mogu biti uspešni u različitim kontekstima.

In [ ]:
![Interakcija Energy-Loudness po klasterima](viz_3_interakcije_klastera.png)

**O grafikonu:**
- Svaki subplot prikazuje jedan klaster
- X osa: Energy, Y osa: Loudness
- Boja tačke: Popularnost (zeleno = visoka, žuto = srednja, crveno = niska)
- Pokazuje kako **kombinacija** dva atributa utiče na popularnost **unutar** svakog klastera
- Različiti klasteri imaju različite obrasce i raspodele popularnosti

### Sumarno: Koja kombinacija karakteristika definiše svaki klaster?

| Klaster | Dominantne VISOKE karakteristike | Dominantne NISKE karakteristike | Prosek Pop. | Top Žanrovi |
|---------|----------------------------------|----------------------------------|-------------|-------------|
| **0 - Metal/Energični** | Energy (+27%)<br>Instrumentalness (+40%)<br>Tempo (+12%) | Acousticness (-79%)<br>Valence (-33%)<br>Loudness (-28%) | **34.1** | Metal, Grindcore |
| **1 - Akustični** | Acousticness (+113%)<br>Loudness (+28%) | Energy (-38%)<br>Instrumentalness (-72%) | **34.2** | Tango, Jazz, Romance |
| **2 - Ambijent** | Instrumentalness (+405%)<br>Acousticness (+158%)<br>Loudness (+146%) | Energy (-68%)<br>Danceability (-35%)<br>Valence (-60%) | **28.6** | Sleep, Classical, Ambient |
| **3 - Plesni** | Danceability (+22%)<br>Valence (+46%)<br>Energy (+15%) | Acousticness (-37%)<br>Instrumentalness (-66%) | **32.9** | Reggaeton, Latino |

 Najpopularnija kombinacija je **akustično + glasno** (Klaster 1), što je kontraintuitivno jer bi se očekivalo da energične pesme dominiraju. Najveći klaster (39% pesama) je **plesna kombinacija** danceability+valence (Klaster 3).

### 2.5 Analiza top 10% najpopularnijih pesama

**Šta čini pesmu veoma popularnom?**

Da bismo identifikovali specifične šablone, analizirali smo top 10% najpopularnijih pesama (popularnost ≥ 63) i uporedili ih sa ostalih 90%.

**Ključne razlike: Top 10% vs Bottom 90%**

| Karakteristika | Top 10% | Bottom 90% | Razlika |
|----------------|---------|------------|---------|
| **Loudness** | -7.19 dB | -8.38 dB | **+14.2%** glasnije |
| **Danceability** | 0.601 | 0.563 | **+6.8%** plesnijih |
| **Instrumentalness** | 0.057 | 0.167 | **-66.2%** manje instrumentala |
| **Acousticness** | 0.255 | 0.322 | **-20.7%** manje akustičnih |
| **Trajanje** | 3.65 min | 3.82 min | **-4.4%** kraće |

**najbolja kombinacija:**
- **Visoka loudness** (> -6 dB) + **Visoka danceability** (> 0.6) + **Niska instrumentalness** (< 0.1)
- Ovaj šablon karakteriše **25.4%** top 10% pesama
- Prosečna popularnost: **72.3** (vs. 70.6 ukupno top 10%)
- **89.9% top 10% ima vokale** (instrumentalness < 0.1)

**Uticaj trajanja pesme:**
- **Najoptimalnija dužina:** 3-4 minuta (46.5% top 10%)
- Top 10% ima kraće pesme: 3.65 min vs. 3.82 min
- **Pesme duže od 6 minuta** imaju -4.4 jedinice niže popularnosti od proseka
- Zaključujemo da današnja publika preferira **kraći sadržaj** (2-4 min = 69% top 10%)

In [ ]:
![Analiza top 10% i trajanja pesama](top_10_analysis.png)

**Interpretacija vizualizacija:**

▪ **Gornji levi grafikon**: Poređenje ključnih karakteristika pokazuje da top 10% ima vidno više vrednosti za loudness i danceability, a dramastično niže za instrumentalness.

▪ **Gornji desni grafikon**: Najpopularnije su pesme dužine **3-4 minuta** i **4-5 minuta** (prosečna popularnost ~34.7), dok su pesme kraće od 2 minuta značajno manje popularne (~28.4).

▪ **Donji levi grafikon**: Scatter plot pokazuje da **nema linearne veze** između trajanja i popularnosti. Većina pesama je koncentrisana između 2-5 minuta, dok pesme preko 7-8 minuta imaju nisku popularnost bez obzira na druge karakteristike (označene tamnije na boja = niži loudness).

▪ **Donji desni grafikon**: Box plot distribucija jasno pokazuje da top 10% ima **signifikantno više vrednosti loudness** i **više vrednosti danceability** (skalirana). Ovo potvrđuje da su glasnije i plesnije pesme statistički dominantnije u top segmentu.

**formula za visoku popularnost** = Glasno + Plesno + Sa vokalima + Kraće (3-4 min)

### 2.6 Uticaj metapodataka: Izvođači i žanrovi

**Da li izvođač ili žanr garantuju popularnost nezavisno od audio karakteristika?**

Iako smo izbacili `artists` i `album_name` za treniranje modela (zbog analize isključivo audio karakteristika), analiza njihovog uticaja otkriva važne uvide o strukturnoj prednosti određenih izvođača i žanrova.

**Ključni nalazi:**

**1. UTICAJ ŽANRA** - Statistički značajan (ANOVA p < 0.001)
- **Žanr objašnjava 25.4% varijanse** u popularnosti
- Razlika između top i bottom žanra: **59.3 vs 2.2** (26× razlika!)

**Top 5 najpopularnijih žanrova:**
| Žanr | Prosečna popularnost | Karakteristike |
|------|---------------------|----------------|
| **pop-film** | 59.3 | Moderate loudness (-7.9 dB), srednja energija |
| **k-pop** | 57.0 | Glasno (-6.5 dB), visoka energija (0.68) |
| **chill** | 53.6 | Tiho (-10.5 dB), niska energija (0.43) |
| **sad** | 52.4 | Tiho (-10.3 dB), niska energija (0.46) |
| **grunge** | 49.6 | Glasno (-5.7 dB), vrlo visoka energija (0.80) |

**Bottom 5 najmanje popularnih žanrova:**
| Žanr | Prosečna popularnost | Karakteristike |
|------|---------------------|----------------|
| **romance** | 3.2 | Vrlo tiho (-13.2 dB), niska energija (0.29) |
| **iranian** | 2.2 | Vrlo tiho (-13.1 dB), srednja energija (0.55) |
| **classical** | 13.1 | Ekstremno tiho (-20.2 dB), najniža energija (0.19) |
| **jazz** | 13.6 | Tiho (-11.6 dB), niska energija (0.35) |
| **grindcore** | 14.6 | Glasno (-6.2 dB), ekstremna energija (0.92) |

**2. UTICAJ IZVOĐAČA** postoji!
- **31,437** jedinstvenih izvođača u datasetu
- Top izvođači imaju **dramatično različite** audio karakteristike od bottom izvođača:

**Poređenje audio karakteristika:**
| Karakteristika | Top 15 izvođača | Bottom 15 izvođača | Razlika |
|----------------|-----------------|-----------------------|---------|
| **Loudness** | -6.2 dB | -10.8 dB | **+42.7%** glasnije |
| **Danceability** | 0.635 | 0.469 | **+35.4%** plesnijih |
| **Energy** | 0.658 | 0.468 | **+40.5%** energičnijih |
| **Instrumentalness** | 0.047 | 0.213 | **-77.8%** više vokala |

**Top 3 po popularnosti (u grupi 15 najproduktivnijih izvođača, 30+ pesama):**
1. **The Beatles** (pop: 61.0, 279 pesama)
2. **Linkin Park** (pop: 56.1, 224 pesama)
3. **Elvis Presley** (pop: 55.1, 169 pesama)

**Napomena o grafikonu metadata_analysis.png:**
- Donji desni graf ne prikazuje sve izvođače sa 30+ pesama, već samo **15 najproduktivnijih** (po broju pesama), uz uslov 30+.
- Označeni "top 3" na tom grafu su top 3 po popularnosti **unutar te grupe od 15**.
- Lista iznad koristi potpuno isti kriterijum kao graf, pa su nazivi identični.

**Zaključak:**  
Iako **žanr garantuje baznu popularnost** (~25% varijanse), **audio karakteristike i brand izvođača** čine preostalih 75%.

In [ ]:
![Analiza metapodataka - Izvođači i žanrovi](metadata_analysis.png)

**Interpretacija vizualizacija:**

▪ **Gornji levi grafikon** — Top 10 žanrova (pop-film, k-pop, chill, sad, grunge) imaju prosečnu popularnost **drastično iznad** ukupnog proseka (crvena linija). Pop-film vodi sa ~59.3, što je skoro 2× više od proseka.

▪ **Gornji desni grafikon** — Bottom 10 žanrova (iranian, romance, classical, jazz) imaju ekstrerno nisku popularnost, veliki deo **ispod 20**. Tradicionalni žanrovi (classical, jazz) pate od niske skorašnje popularnosti na streaming platformama.

▪ **Donji levi grafikon** — Box plot distribucija top 5 žanrova pokazuje da pop-film i k-pop imaju **najmanju varijansu** (konzistentno visoka popularnost), dok chill i sad imaju širi raspon. Medijane su visoko pozicionirane.

▪ **Donji desni grafikon** — Scatter plot produktivnosti vs popularnosti izvođača (sa 30+ pesama) pokazuje da **broj pesama NE garantuje popularnost**. Neki izvođači sa 100+ pesama imaju nisku prosečnu popularnost (~20-30), dok drugi sa ~50 pesama dosežu 70-80+. Boja označava popularnost - zelene tačke (top performers) su ravnomerno raspoređene bez obzira na produktivnost.

**Ključni uvid:**  
Žanr postavlja baznu popularnost** (~25% varijanse), ali **izvođači sa brand reputacijom i konzistentnim audio kvalitetom** nadmašuju žanrovske proseke.

## 3. Metodologija

Na osnovu analize literature i karakteristika problema, odabrali smo **regresioni pristup** za predikciju tačne vrednosti popularnosti.

### Odabrani algoritmi

**Random Forest Regressor**
- Ensemble metoda bazirana na više stabala odlučivanja
- Robusna na outliers i ne zahteva feature scaling
- Otporna na overfitting
- Efikasna pri radu sa većim brojem ulaznih karakteristika
- Parametri: 100 stabala, maksimalna dubina 20

**XGBoost Regressor**
- Gradient boosting algoritam
- Sekvencijalno gradi model gde svaki novi model ispravlja greške prethodnog
- Brz i efikasan za velike datasete
- Parametri: 100 stabala, learning rate 0.1, maksimalna dubina 6

### Metrike evaluacije

- **Mean Absolute Error (MAE)** - prosečna apsolutna greška
- **R² Score** - koeficijent determinacije, pokazuje koliko model objašnjava varijansu podataka

**Podela podataka:** 80% training, 20% test

In [ ]:
# Priprema podataka
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Audio features + enkodovanje kategoričkih varijabli
audio_features = ['danceability', 'energy', 'loudness', 'acousticness', 
                  'instrumentalness', 'valence', 'tempo', 'duration_ms']
df['genre_encoded'] = LabelEncoder().fit_transform(df['track_genre'])

X = df[audio_features + ['genre_encoded']]
y = df['popularity']

# Podela podataka
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {len(X_train):,} | Test: {len(X_test):,} pesama")

In [ ]:
# Treniranje Random Forest i XGBoost modela
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42)
rf_model.fit(X_train, y_train)

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

print("Modeli uspešno trenirani")

## 4. Rezultati i performanse

In [ ]:
# Evaluacija performansi
from sklearn.metrics import mean_absolute_error, r2_score

rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

print("Random Forest:")
print(f"  MAE: {mean_absolute_error(y_test, rf_pred):.3f}")
print(f"  R²:  {r2_score(y_test, rf_pred):.3f}")

print("\nXGBoost:")
print(f"  MAE: {mean_absolute_error(y_test, xgb_pred):.3f}")
print(f"  R²:  {r2_score(y_test, xgb_pred):.3f}")

In [ ]:
![Važnost atributa](viz_4_feature_importance.png)

In [ ]:
![Greška predikcije po klasterima](viz_5_mae_po_klasterima.png)

## 4.5. Analiza kvaliteta podataka: Rešavanje problema sa nulama

### Problem

Tokom analize dataset-a, otkriveno je da **16,019 pesama (14.1%)** ima vrednost popularnosti označenu kao 0. Ovo postavlja pitanje: da li su te pesme zaista potpuno nepopularne, ili se radi o nedostajućim vrednostima?

### Poređenje audio karakteristika

Analiza pokazuje da pesme sa popularnost=0 imaju slične audio karakteristike kao i pesme sa pop>0, što sugeriše da su verovatno nedostajuće vrednosti, a ne potpuno nepopularne pesme.

### Eksperimenti: Dva pristupa

**Eksperiment 1: Uklanjanje pesama sa pop=0**
- Dataset veličina: 97,980 pesama (gubitak 14.1% podataka)
- Treniranje Random Forest modela
- **Rezultati**: MAE = 10.114, R² = 0.477

**Eksperiment 2: KNN Imputation**
- Pristup: Pronalaženje 10 najsličnijih pesama (na osnovu audio karakteristika) i dodeljivanje prosečne popularnosti
- Dataset veličina: 113,999 pesama (zadržani svi podaci)
- Prosečna imputirana vrednost: 37.27
- **Rezultati**: MAE = 9.747, R² = 0.467

In [ ]:
![Poređenje pristupa za rešavanje problem nula](zero_popularity_comparison.png)

### Zaključci analize

**Ključni nalazi:**

1. **KNN Imputation je bolji pristup** - niži MAE uprkos blago nižem R²
   - Poboljšanje MAE: **3.63%** (10.114 → 9.747)
   - Zadržavanje svih 113,999 podataka (bez gubitka 14.1% dataset-a)

2. **Priroda nula**: Analiza audio karakteristika pokazuje da pesme sa pop=0 nisu bitno drugačije od ostalih, što sugeriše da je reč o nedostajućim vrednostima, a ne o zaista nepopularnim pesmama

3. **Distribucija imputiranih vrednosti**: Imputirane vrednosti (mean=37.27, median=38.20) su realne i odgovaraju distribuciji popularnosti u dataset-u

Korisicemo KNN Imputation pristup za finalni model jer:
- Daje bolje performanse (niži MAE)
- Zadržava sve podatke
- Koristi sličnost audio karakteristika za pametno popunjavanje nedostajućih vrednosti

**Dataset-ovi sačuvani za dalja istraživanja:**
- `dataset_removed_zeros.csv` (bez pop=0, 97,980 pesama)
- `dataset_knn_imputed.csv` (sa KNN imputation, 113,999 pesama)

## 5. Zaključci i diskusija

### Performanse modela

Finalni modeli (sa KNN Imputation pristupom) pokazuju **solidne performanse**:
- **MAE ~ 9.75**: U proseku, predikcija greši za ~9.75 poena (poboljšanje od 3.63% u odnosu na uklanjanje nula)
- **R² ~ 0.467**: Modeli objašnjavaju ~47% varijanse u popularnosti
- Random Forest i XGBoost postižu slične rezultate

### Kvalitet podataka

**Ključno otkriće**: 14.1% dataset-a (16,019 pesama) ima popularnost=0
- Analiza pokazuje da su verovatno **nedostajuće vrednosti**, ne potpuno nepopularne pesme
- **KNN Imputation pristup** pokazuje bolje rezultate od jednostavnog uklanjanja
- Korišćenje sličnosti audio karakteristika omogućava pametno popunjavanje podataka

### Najvažniji faktori

Analiza važnosti atributa pokazuje da najveći uticaj imaju:
1. **Loudness** (glasnoća) - najjača pozitivna korelacija
2. **Genre** (žanr) - različiti žanrovi imaju različite nivoe popularnosti
3. **Energy, Danceability** - energetske karakteristike pesme
4. **Duration_ms** - trajanje pesme

### Dubinski šabloni i klasteri

Klaster analiza je otkrila važne šablone koji prevazilaze jednostavne korelacije:

**Pronađeni klasteri:**
- Pesame se prirodno grupišu u 4 distinktna klastera sa različitim audio profilima
- **Energične pesme** (visoka energy, loudness) imaju najveću prosečnu popularnost
- **Akustične pesme** formiraju poseban klaster sa umerenom popularnošću
- **Instrumentalne pesme** pokazuju najnižu popularnost i najjasniju separaciju

**Važno:**
- Popularnost zavisi od **kombinacije karakteristika**, ne pojedinačnih atributa
- Postoje **različiti šabloni uspešnosti** - nema jedinstvenog "recepta"
- Interakcije između atributa (npr. energy + loudness) imaju snažniji uticaj od bilo kojeg pojedinačnog atributa

### Ključna saznanja

1. **Kvalitet podataka je kritičan**: Pametno rešavanje nedostajućih vrednosti (KNN Imputation) poboljšava performanse za 3.63%

2. Audio karakteristike **objašnjavaju skoro 50% varijanse** u popularnosti - značajno bolji rezultat od očekivanog (15-18% iz literature)

3. **Glasnije pesme** su u proseku popularnije

4. **Žanr ima veliki uticaj** (ANOVA test, p < 0.001)

5. Većina pesama ima **nisku popularnost** (<25), distribucija je neujednačena

6. **Šabloni i interakcije** između atributa su važniji od pojedinačnih korelacija

**Finalni zaključak**: Random Forest uz KNN Imputation pristup je pouzdan algoritam za ovaj problem i postiže rezultate značajno bolje od očekivanih iz literature.

